## Cell 1: Setup & Imports


In [ ]:
import os
import json
import glob
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, cohen_kappa_score

## Cell 2: Define Base Paths

In [ ]:
# Base directory for Multi-Task-Learning results
BASE = Path(__file__).parent if '__file__' in dir() else Path.cwd()

# Experiment paths (using relative paths for public repo)
RESULTS_BASELINE = BASE / "results" / "baseline"
RESULTS_SKILLS = BASE / "results" / "skills_variant"

# Auto-detect available folds
baseline_folders = sorted([d for d in RESULTS_BASELINE.glob("fold_*") if d.is_dir()])
folds_available = [int(d.name.split('_')[1]) for d in baseline_folders]

print(f"Base path: {BASE}")
print(f"Available folds: {folds_available}")
print(f"Baseline results: {RESULTS_BASELINE}")
print(f"Skills variant results: {RESULTS_SKILLS}")

## Cell 3: Load Metrics from All Folds

In [ ]:
# Metrics to focus on
METRICS_OF_INTEREST = ['eval_f1_macro', 'eval_f1_weighted', 'eval_mae', 'eval_qwk']

def load_test_metrics(results_dir, folds):
    """Load test metrics from all folds."""
    metrics_data = []
    
    for fold in folds:
        metrics_file = results_dir / f"fold_{fold}" / "predictions" / "test_metrics.json"
        
        if not metrics_file.exists():
            print(f"Warning: {metrics_file} not found")
            continue
        
        try:
            with open(metrics_file, 'r') as f:
                metrics = json.load(f)
            
            row = {'fold': fold}
            for metric in METRICS_OF_INTEREST:
                if metric in metrics:
                    row[metric] = metrics[metric]
            
            metrics_data.append(row)
        except Exception as e:
            print(f"Error loading {metrics_file}: {e}")
    
    return pd.DataFrame(metrics_data)

# Load metrics for both methods
print("\n" + "="*60)
print("Loading Test Metrics from All Folds")
print("="*60)

baseline_metrics = load_test_metrics(RESULTS_BASELINE, folds_available)
skills_metrics = load_test_metrics(RESULTS_SKILLS, folds_available)

print(f"\n✅ Baseline metrics shape: {baseline_metrics.shape}")
print(baseline_metrics)

print(f"\n✅ Skills variant metrics shape: {skills_metrics.shape}")
print(skills_metrics)

## Cell 4: Aggregate Metrics (Mean ± Std)

In [ ]:
# Create display names for metrics
metric_display_names = {
    'eval_f1_macro': 'F1-Macro',
    'eval_f1_weighted': 'F1-Weighted',
    'eval_mae': 'MAE',
    'eval_qwk': 'QWK',
}

def aggregate_metrics(df, method_name):
    """Calculate mean and std for each metric."""
    if df.empty:
        return None
    
    agg_results = {'Method': method_name}
    
    for metric in METRICS_OF_INTEREST:
        if metric in df.columns:
            mean_val = df[metric].mean()
            std_val = df[metric].std()
            metric_name = metric_display_names.get(metric, metric)
            agg_results[f"{metric_name} (mean)"] = mean_val
            agg_results[f"{metric_name} (std)"] = std_val
    
    return agg_results

# Aggregate results
print("\n" + "="*80)
print("AGGREGATED TEST METRICS (Mean ± Std across all folds)")
print("="*80)

baseline_agg = aggregate_metrics(baseline_metrics, "Baseline")
skills_agg = aggregate_metrics(skills_metrics, "Skills Variant")

# Create summary table
summary_df = pd.DataFrame([baseline_agg, skills_agg])

# Format and display
pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print("\n")
display(summary_df)

# Save summary
summary_df.to_csv("metrics_summary.csv", index=False)
print("\n✅ Saved summary to: metrics_summary.csv")

## Cell 5: Per-Fold Comparison Table

In [ ]:
# Merge baseline and skills metrics by fold for side-by-side comparison
merged_metrics = baseline_metrics.copy()
merged_metrics = merged_metrics.rename(columns={
    'eval_f1_macro': 'Baseline_F1-Macro',
    'eval_f1_weighted': 'Baseline_F1-Weighted',
    'eval_mae': 'Baseline_MAE',
    'eval_qwk': 'Baseline_QWK',
})

skills_rename = {
    'eval_f1_macro': 'Skills_F1-Macro',
    'eval_f1_weighted': 'Skills_F1-Weighted',
    'eval_mae': 'Skills_MAE',
    'eval_qwk': 'Skills_QWK',
}
skills_metrics_renamed = skills_metrics.rename(columns=skills_rename)

merged_metrics = merged_metrics.merge(skills_metrics_renamed[['fold'] + list(skills_rename.values())], 
                                      on='fold', how='outer')

print("\n" + "="*80)
print("PER-FOLD METRICS COMPARISON")
print("="*80)
print("\n")
display(merged_metrics.round(4))

# Save per-fold comparison
merged_metrics.to_csv("per_fold_metrics.csv", index=False)
print("\n✅ Saved per-fold comparison to: per_fold_metrics.csv")

## Cell 6: Helper Functions for Statistical Tests

In [ ]:
def hedges_g_repeated_measures(method1_scores, method2_scores):
    """
    Calculate Hedges' g for repeated measures design.
    Using Goulet-Pelletier & Cousineau (2018) method.
    """
    m1 = np.asarray(method1_scores, dtype=float)
    m2 = np.asarray(method2_scores, dtype=float)
    
    n = len(m1)
    
    # Pooled standard deviation
    s1 = np.std(m1, ddof=1)
    s2 = np.std(m2, ddof=1)
    S_p = np.sqrt((s1**2 + s2**2) / 2)
    
    # Cohen's d
    M1 = np.mean(m1)
    M2 = np.mean(m2)
    cohens_d = (M1 - M2) / S_p if S_p > 0 else 0
    
    # Correction factor J
    nu = 2 * (n - 1)
    J = 1 - (3 / (4 * nu - 1))
    
    # Hedges' g
    hedges_g = cohens_d * J
    
    return {
        "M1": M1,
        "M2": M2,
        "S_p": S_p,
        "s1": s1,
        "s2": s2,
        "cohens_d": cohens_d,
        "hedges_g": hedges_g,
        "mean_diff": M1 - M2,
        "nu": nu,
        "J": J,
    }


def paired_ttest_with_hedges_g(baseline_scores, skills_scores, metric_name):
    """
    Perform paired t-test and calculate Hedges' g effect size.
    Returns detailed statistics for the comparison.
    """
    a = np.asarray(baseline_scores, dtype=float)
    b = np.asarray(skills_scores, dtype=float)
    
    n = len(a)
    if n < 2:
        return None
    
    # Calculate differences
    diffs = a - b
    mean_diff = np.mean(diffs)
    std_diff = np.std(diffs, ddof=1)
    se = std_diff / np.sqrt(n)
    
    # Paired t-test
    t_stat, p_val = stats.ttest_rel(a, b)
    
    # Calculate Hedges' g
    hedges_result = hedges_g_repeated_measures(a, b)
    
    # 95% CI for mean difference
    ci = stats.t.interval(0.95, df=n-1, loc=mean_diff, scale=se)
    
    # Effect size category
    abs_g = abs(hedges_result["hedges_g"])
    if abs_g < 0.2:
        effect_cat = "negligible"
    elif abs_g < 0.5:
        effect_cat = "small"
    elif abs_g < 0.8:
        effect_cat = "medium"
    else:
        effect_cat = "large"
    
    return {
        "metric": metric_name,
        "t_statistic": t_stat,
        "p_value": p_val,
        "mean_diff": mean_diff,
        "ci_low": ci[0],
        "ci_high": ci[1],
        "S_p": hedges_result["S_p"],
        "S1": hedges_result["s1"],
        "S2": hedges_result["s2"],
        "cohens_d": hedges_result["cohens_d"],
        "hedges_g": hedges_result["hedges_g"],
        "correction_factor_J": hedges_result["J"],
        "degrees_of_freedom": hedges_result["nu"],
        "effect_size_category": effect_cat,
        "n_pairs": n,
        "significant_p05": p_val < 0.05,
    }

print("✅ Statistical test functions loaded")

## Cell 7: Paired T-Tests (Baseline vs Skills Variant)

In [ ]:
print("\n" + "="*80)
print("PAIRED T-TEST ANALYSIS: Baseline vs Skills Variant")
print("(Repeated Measures Design with Hedges' g Effect Size)")
print("="*80)

# Ensure both dataframes have the same folds
common_folds = sorted(set(baseline_metrics['fold']) & set(skills_metrics['fold']))

if len(common_folds) < 2:
    print("⚠️  Not enough common folds for pairing (min: 2)")
else:
    # Filter to common folds only
    baseline_paired = baseline_metrics[baseline_metrics['fold'].isin(common_folds)].sort_values('fold')
    skills_paired = skills_metrics[skills_metrics['fold'].isin(common_folds)].sort_values('fold')
    
    print(f"\n📊 Paired folds: {common_folds}")
    print(f"Number of pairs (n): {len(common_folds)}\n")
    
    # Run t-tests for each metric
    ttest_results = []
    
    for metric in METRICS_OF_INTEREST:
        if metric not in baseline_paired.columns or metric not in skills_paired.columns:
            continue
        
        baseline_vals = baseline_paired[metric].values
        skills_vals = skills_paired[metric].values
        
        metric_display_name = metric_display_names.get(metric, metric)
        
        result = paired_ttest_with_hedges_g(baseline_vals, skills_vals, metric_display_name)
        
        if result:
            ttest_results.append(result)
            
            # Print results
            print(f"\n{'='*70}")
            print(f"📈 {metric_display_name}")
            print(f"{'='*70}")
            print(f"t-statistic:        {result['t_statistic']:>8.4f}")
            print(f"p-value:            {result['p_value']:>8.4f} {'✅ Significant' if result['significant_p05'] else '❌ Not significant'}")
            print(f"Mean Difference:    {result['mean_diff']:>+8.4f}  (Baseline - Skills)")
            print(f"95% CI:             [{result['ci_low']:>+8.4f}, {result['ci_high']:>+8.4f}]")
            print(f"\nEffect Size:")
            print(f"  Pooled SD (S_p):  {result['S_p']:>8.4f}")
            print(f"  Cohen's d:        {result['cohens_d']:>+8.4f}")
            print(f"  Hedges' g:        {result['hedges_g']:>+8.4f}  ({result['effect_size_category'].upper()})")
            print(f"  Correction J:     {result['correction_factor_J']:>8.4f}")
            print(f"  DF (ν):           {int(result['degrees_of_freedom'])}")
    
    # Create results dataframe
    ttest_df = pd.DataFrame(ttest_results)
    
    print("\n" + "="*80)
    print("SUMMARY TABLE: Paired T-Test Results")
    print("="*80)
    print("\n")
    
    # Format for display
    display_cols = ['metric', 't_statistic', 'p_value', 'mean_diff', 'hedges_g', 
                    'effect_size_category', 'significant_p05']
    display(ttest_df[display_cols].round(4))
    
    # Save results
    ttest_df.to_csv("paired_ttest_results.csv", index=False)
    print("\n✅ Saved paired t-test results to: paired_ttest_results.csv")